In [1]:
import pandas as pd
import numpy as np

In [2]:
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')
m_seeds = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

In [3]:
m_seeds

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
2621,2025,Z12,1161
2622,2025,Z13,1213
2623,2025,Z14,1423
2624,2025,Z15,1303


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')


def build_team_features(df):
   
    win_cols = {
        'Season': 'Season', 'WTeamID': 'TeamID',
        'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3': 'FGM3', 'WFGA3': 'FGA3',
        'WFTM': 'FTM', 'WFTA': 'FTA', 'WOR': 'OR', 'WDR': 'DR',
        'WAst': 'Ast', 'WTO': 'TO', 'WStl': 'Stl', 'WBlk': 'Blk',
        'WPF': 'PF', 'WScore': 'PointsFor', 'LScore': 'PointsAgainst',
        'NumOT': 'NumOT'
    }
    win_df = df[list(win_cols.keys())].rename(columns=win_cols)
    win_df['Win'] = 1

    # --- Losing team perspective ---
    loss_cols = {
        'Season': 'Season', 'LTeamID': 'TeamID',
        'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3', 'LFGA3': 'FGA3',
        'LFTM': 'FTM', 'LFTA': 'FTA', 'LOR': 'OR', 'LDR': 'DR',
        'LAst': 'Ast', 'LTO': 'TO', 'LStl': 'Stl', 'LBlk': 'Blk',
        'LPF': 'PF', 'LScore': 'PointsFor', 'WScore': 'PointsAgainst',
        'NumOT': 'NumOT'
    }
    loss_df = df[list(loss_cols.keys())].rename(columns=loss_cols)
    loss_df['Win'] = 0

    combined = pd.concat([win_df, loss_df], ignore_index=True)

    # Derived per-game stats
    combined['FG_pct']  = combined['FGM']  / combined['FGA'].replace(0, np.nan)
    combined['FG3_pct'] = combined['FGM3'] / combined['FGA3'].replace(0, np.nan)
    combined['FT_pct']  = combined['FTM']  / combined['WFTA'] if 'WFTA' in combined else (
                          combined['FTM']  / combined['FTA'].replace(0, np.nan))
    combined['FT_pct']  = combined['FTM']  / combined['FTA'].replace(0, np.nan)
    combined['PointDiff'] = combined['PointsFor'] - combined['PointsAgainst']

    agg = combined.groupby(['Season', 'TeamID']).agg(
        Games        = ('Win',          'count'),
        WinRate      = ('Win',          'mean'),
        AvgPtsFor    = ('PointsFor',    'mean'),
        AvgPtsAgainst= ('PointsAgainst','mean'),
        AvgPointDiff = ('PointDiff',    'mean'),
        AvgFGpct     = ('FG_pct',       'mean'),
        AvgFG3pct    = ('FG3_pct',      'mean'),
        AvgFTpct     = ('FT_pct',       'mean'),
        AvgOR        = ('OR',           'mean'),
        AvgDR        = ('DR',           'mean'),
        AvgAst       = ('Ast',          'mean'),
        AvgTO        = ('TO',           'mean'),
        AvgStl       = ('Stl',          'mean'),
        AvgBlk       = ('Blk',          'mean'),
    ).reset_index()

    return agg

team_features = build_team_features(m_regular_season_detailed)
print("\nTeam features shape:", team_features.shape)
print(team_features.head(3))

# ─────────────────────────────────────────
# 3. PARSE SEEDS  →  numeric seed value
# ─────────────────────────────────────────

def parse_seed(seed_str):
    """'W01' → 1,  'Z14a' → 14"""
    return int(''.join(filter(str.isdigit, str(seed_str))))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

# ─────────────────────────────────────────
# 4. BUILD MATCHUP FEATURES
#    For every tourney game produce one row:
#    feature = team1_stats - team2_stats
#    label   = 1 if team1 won, 0 otherwise
# ─────────────────────────────────────────

FEATURE_COLS = [
    'WinRate', 'AvgPtsFor', 'AvgPtsAgainst', 'AvgPointDiff',
    'AvgFGpct', 'AvgFG3pct', 'AvgFTpct',
    'AvgOR', 'AvgDR', 'AvgAst', 'AvgTO', 'AvgStl', 'AvgBlk',
    'SeedNum'
]

def build_matchup_df(tourney_df, team_feat, seeds):
    rows = []
    for _, game in tourney_df.iterrows():
        season  = game['Season']
        t1, t2  = game['WTeamID'], game['LTeamID']

        f1 = team_feat[(team_feat.Season == season) & (team_feat.TeamID == t1)]
        f2 = team_feat[(team_feat.Season == season) & (team_feat.TeamID == t2)]
        s1 = seeds[(seeds.Season == season) & (seeds.TeamID == t1)]
        s2 = seeds[(seeds.Season == season) & (seeds.TeamID == t2)]

        if f1.empty or f2.empty or s1.empty or s2.empty:
            continue

        f1 = f1.iloc[0]; f2 = f2.iloc[0]
        s1 = s1.iloc[0]['SeedNum']; s2 = s2.iloc[0]['SeedNum']

        diff = {}
        for col in FEATURE_COLS[:-1]:          # all except SeedNum
            diff[f'diff_{col}'] = f1[col] - f2[col]
        diff['diff_SeedNum'] = s1 - s2         # lower seed = better → neg diff is good

        diff['Season']  = season
        diff['Team1']   = t1
        diff['Team2']   = t2
        diff['Label']   = 1   # team1 (winner) always goes first

        # Mirror the game (swap teams) to avoid positional bias
        diff2 = {k: -v if k.startswith('diff_') else v for k, v in diff.items()}
        diff2['Team1'] = t2; diff2['Team2'] = t1; diff2['Label'] = 0

        rows.extend([diff, diff2])

    return pd.DataFrame(rows)

matchup_df = build_matchup_df(m_tourney_detailed_result, team_features, m_seeds)
print("\nMatchup dataset shape:", matchup_df.shape)
print("Label distribution:\n", matchup_df['Label'].value_counts())

# ─────────────────────────────────────────
# 5. TRAIN / TEST SPLIT
#    Train → 2005, 2006, 2007
#    Test  → 2008
# ─────────────────────────────────────────

DIFF_COLS = [c for c in matchup_df.columns if c.startswith('diff_')]

train = matchup_df[matchup_df['Season'].isin([2005, 2006, 2007])]
test  = matchup_df[matchup_df['Season'] == 2008]

X_train, y_train = train[DIFF_COLS], train['Label']
X_test,  y_test  = test[DIFF_COLS],  test['Label']

print(f"\nTrain size: {len(X_train)} rows | Test size: {len(X_test)} rows")

# ─────────────────────────────────────────
# 6. TRAIN RANDOM FOREST
# ─────────────────────────────────────────

rf = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = 6,       # keep shallow to avoid overfitting small dataset
    min_samples_leaf = 5,
    max_features     = 'sqrt',
    random_state     = 42,
    n_jobs           = -1
)
rf.fit(X_train, y_train)

# ─────────────────────────────────────────
# 7. EVALUATE ON 2008
# ─────────────────────────────────────────

y_pred_proba = rf.predict_proba(X_test)[:, 1]
y_pred_class = (y_pred_proba >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred_class)
ll   = log_loss(y_test, y_pred_proba)

print(f"\n── 2008 Evaluation ──")
print(f"Accuracy : {acc:.4f}")
print(f"Log Loss : {ll:.4f}   (Kaggle metric – lower is better, baseline ~0.693)")

# ─────────────────────────────────────────
# 8. FEATURE IMPORTANCE
# ─────────────────────────────────────────

importance_df = pd.DataFrame({
    'Feature'   : DIFF_COLS,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n── Top 10 Features ──")
print(importance_df.head(10).to_string(index=False))

# ─────────────────────────────────────────
# 9. GENERATE KAGGLE SUBMISSION FORMAT
#    All possible 2008 matchups (team1 < team2)
# ─────────────────────────────────────────

teams_2008 = m_seeds[m_seeds['Season'] == 2008]['TeamID'].unique()
sub_rows = []

for i, t1 in enumerate(sorted(teams_2008)):
    for t2 in sorted(teams_2008)[i+1:]:
        f1 = team_features[(team_features.Season == 2008) & (team_features.TeamID == t1)]
        f2 = team_features[(team_features.Season == 2008) & (team_features.TeamID == t2)]
        s1 = m_seeds[(m_seeds.Season == 2008) & (m_seeds.TeamID == t1)]
        s2 = m_seeds[(m_seeds.Season == 2008) & (m_seeds.TeamID == t2)]

        if f1.empty or f2.empty or s1.empty or s2.empty:
            continue

        f1 = f1.iloc[0]; f2 = f2.iloc[0]
        s1n = s1.iloc[0]['SeedNum']; s2n = s2.iloc[0]['SeedNum']

        row = {}
        for col in FEATURE_COLS[:-1]:
            row[f'diff_{col}'] = f1[col] - f2[col]
        row['diff_SeedNum'] = s1n - s2n

        prob = rf.predict_proba(pd.DataFrame([row])[DIFF_COLS])[0, 1]
        sub_rows.append({'ID': f'2008_{t1}_{t2}', 'Pred': prob})

submission = pd.DataFrame(sub_rows)
submission.to_csv('/mnt/user-data/outputs/submission_2008.csv', index=False)
print(f"\n✅ Submission saved → submission_2008.csv  ({len(submission)} matchups)")
print(submission.head(10))


Team features shape: (8346, 16)
   Season  TeamID  Games   WinRate  AvgPtsFor  AvgPtsAgainst  AvgPointDiff  \
0    2003    1102     28  0.428571  57.250000      57.000000      0.250000   
1    2003    1103     27  0.481481  78.777778      78.148148      0.629630   
2    2003    1104     28  0.607143  69.285714      65.000000      4.285714   

   AvgFGpct  AvgFG3pct  AvgFTpct      AvgOR      AvgDR     AvgAst      AvgTO  \
0  0.486149   0.367637  0.642402   4.178571  16.821429  13.000000  11.428571   
1  0.487294   0.331990  0.735271   9.777778  19.925926  15.222222  12.629630   
2  0.419676   0.325442  0.705168  13.571429  23.928571  12.107143  13.285714   

     AvgStl    AvgBlk  
0  5.964286  1.785714  
1  7.259259  2.333333  
2  6.607143  3.785714  

Matchup dataset shape: (2898, 18)
Label distribution:
 Label
1    1449
0    1449
Name: count, dtype: int64

Train size: 384 rows | Test size: 128 rows

── 2008 Evaluation ──
Accuracy : 0.7578
Log Loss : 0.4843   (Kaggle metric – lower i

OSError: Cannot save file into a non-existent directory: '/mnt/user-data/outputs'

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score
import warnings
warnings.filterwarnings('ignore')

def parse_seed(seed_str):
    """'W01' → 1,  'Z14a' → 14"""
    return int(''.join(filter(str.isdigit, str(seed_str))))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

# ─────────────────────────────────────────
# 3. BUILD TEAM SEASON FEATURES
#    Uses FULL regular season stats per team
# ─────────────────────────────────────────
def build_team_features(df):
    """
    Stack winner + loser rows so every game contributes
    one record per team, then aggregate per (Season, TeamID).
    """
    win_df = df.rename(columns={
        'WTeamID':'TeamID','WScore':'PtsFor','LScore':'PtsAgainst',
        'WFGM':'FGM','WFGA':'FGA','WFGM3':'FGM3','WFGA3':'FGA3',
        'WFTM':'FTM','WFTA':'FTA','WOR':'OR','WDR':'DR',
        'WAst':'Ast','WTO':'TO','WStl':'Stl','WBlk':'Blk','WPF':'PF'
    })[['Season','TeamID','PtsFor','PtsAgainst',
        'FGM','FGA','FGM3','FGA3','FTM','FTA',
        'OR','DR','Ast','TO','Stl','Blk','PF']].copy()
    win_df['Win'] = 1

    loss_df = df.rename(columns={
        'LTeamID':'TeamID','LScore':'PtsFor','WScore':'PtsAgainst',
        'LFGM':'FGM','LFGA':'FGA','LFGM3':'FGM3','LFGA3':'FGA3',
        'LFTM':'FTM','LFTA':'FTA','LOR':'OR','LDR':'DR',
        'LAst':'Ast','LTO':'TO','LStl':'Stl','LBlk':'Blk','LPF':'PF'
    })[['Season','TeamID','PtsFor','PtsAgainst',
        'FGM','FGA','FGM3','FGA3','FTM','FTA',
        'OR','DR','Ast','TO','Stl','Blk','PF']].copy()
    loss_df['Win'] = 0

    combined = pd.concat([win_df, loss_df], ignore_index=True)
    combined['FG_pct']    = combined['FGM']  / combined['FGA'].replace(0, np.nan)
    combined['FG3_pct']   = combined['FGM3'] / combined['FGA3'].replace(0, np.nan)
    combined['FT_pct']    = combined['FTM']  / combined['FTA'].replace(0, np.nan)
    combined['PointDiff'] = combined['PtsFor'] - combined['PtsAgainst']

    agg = combined.groupby(['Season','TeamID']).agg(
        Games         = ('Win',        'count'),
        WinRate       = ('Win',        'mean'),
        AvgPtsFor     = ('PtsFor',     'mean'),
        AvgPtsAgainst = ('PtsAgainst', 'mean'),
        AvgPointDiff  = ('PointDiff',  'mean'),
        AvgFGpct      = ('FG_pct',     'mean'),
        AvgFG3pct     = ('FG3_pct',    'mean'),
        AvgFTpct      = ('FT_pct',     'mean'),
        AvgOR         = ('OR',         'mean'),
        AvgDR         = ('DR',         'mean'),
        AvgAst        = ('Ast',        'mean'),
        AvgTO         = ('TO',         'mean'),
        AvgStl        = ('Stl',        'mean'),
        AvgBlk        = ('Blk',        'mean'),
    ).reset_index()
    return agg

team_features = build_team_features(m_regular_season_detailed)
print("\nTeam features shape:", team_features.shape)

# ─────────────────────────────────────────
# 4. BUILD MATCHUP DATASET
#    Source: ALL tournament games (not just 3 seasons)
#    Each game → 2 mirrored rows to remove positional bias
# ─────────────────────────────────────────
STAT_COLS = [
    'WinRate','AvgPtsFor','AvgPtsAgainst','AvgPointDiff',
    'AvgFGpct','AvgFG3pct','AvgFTpct',
    'AvgOR','AvgDR','AvgAst','AvgTO','AvgStl','AvgBlk',
    'SeedNum'
]

def build_matchup_df(tourney_df, team_feat, seeds):
    # Pre-index for speed
    tf_idx = team_feat.set_index(['Season','TeamID'])
    sd_idx = seeds.set_index(['Season','TeamID'])

    rows = []
    skipped = 0
    for _, game in tourney_df.iterrows():
        season = game['Season']
        t1, t2 = game['WTeamID'], game['LTeamID']

        try:
            f1 = tf_idx.loc[(season, t1)]
            f2 = tf_idx.loc[(season, t2)]
            s1 = sd_idx.loc[(season, t1), 'SeedNum']
            s2 = sd_idx.loc[(season, t2), 'SeedNum']
        except KeyError:
            skipped += 1
            continue

        def make_row(team_a, team_b, fa, fb, sa, sb, label):
            r = {'Season': season, 'Team1': team_a, 'Team2': team_b, 'Label': label}
            for col in STAT_COLS[:-1]:
                r[f'diff_{col}'] = fa[col] - fb[col]
            r['diff_SeedNum'] = sa - sb
            return r

        rows.append(make_row(t1, t2, f1, f2, s1, s2, label=1))
        rows.append(make_row(t2, t1, f2, f1, s2, s1, label=0))

    print(f"Skipped {skipped} games (missing team/seed data)")
    return pd.DataFrame(rows)

matchup_df = build_matchup_df(m_tourney_detailed_result, team_features, m_seeds)
DIFF_COLS  = [c for c in matchup_df.columns if c.startswith('diff_')]

print(f"\nTotal matchup rows    : {len(matchup_df)}")
print(f"Seasons in matchup df : {sorted(matchup_df['Season'].unique())}")
print(f"Label distribution    :\n{matchup_df['Label'].value_counts()}")

# ─────────────────────────────────────────
# 5. TRAIN / TEST SPLIT
#    Train → everything BEFORE 2008
#    Test  → 2008 tournament
# ─────────────────────────────────────────
train = matchup_df[matchup_df['Season'] < 2008]
test  = matchup_df[matchup_df['Season'] == 2008]

X_train, y_train = train[DIFF_COLS], train['Label']
X_test,  y_test  = test[DIFF_COLS],  test['Label']

print(f"\nTrain rows : {len(X_train)}  (seasons {sorted(train['Season'].unique())})")
print(f"Test rows  : {len(X_test)}   (season 2008)")

# ─────────────────────────────────────────
# 6. TRAIN RANDOM FOREST
# ─────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators     = 500,
    max_depth        = 6,
    min_samples_leaf = 5,
    max_features     = 'sqrt',
    class_weight     = 'balanced',
    random_state     = 42,
    n_jobs           = -1
)
rf.fit(X_train, y_train)
print("\nModel trained ✅")

# ─────────────────────────────────────────
# 7. EVALUATE ON 2008 TOURNAMENT
# ─────────────────────────────────────────
y_pred_proba = rf.predict_proba(X_test)[:, 1]
y_pred_class = (y_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred_class)
ll  = log_loss(y_test, y_pred_proba)

print(f"\n── 2008 Tournament Evaluation ──")
print(f"Accuracy  : {acc:.4f}")
print(f"Log Loss  : {ll:.4f}  (Kaggle metric — baseline random = 0.693)")

# ─────────────────────────────────────────
# 8. FEATURE IMPORTANCE
# ─────────────────────────────────────────
importance_df = (
    pd.DataFrame({'Feature': DIFF_COLS, 'Importance': rf.feature_importances_})
    .sort_values('Importance', ascending=False)
)
print("\n── Top 10 Features ──")
print(importance_df.head(10).to_string(index=False))

# ─────────────────────────────────────────
# 9. GENERATE KAGGLE SUBMISSION
#    All possible 2008 matchups (t1 < t2)
# ─────────────────────────────────────────
tf_idx = team_features.set_index(['Season','TeamID'])
sd_idx = m_seeds.set_index(['Season','TeamID'])

teams_2008 = sorted(m_seeds[m_seeds['Season'] == 2008]['TeamID'].unique())
sub_rows   = []

for i, t1 in enumerate(teams_2008):
    for t2 in teams_2008[i+1:]:
        try:
            f1  = tf_idx.loc[(2008, t1)]
            f2  = tf_idx.loc[(2008, t2)]
            s1n = sd_idx.loc[(2008, t1), 'SeedNum']
            s2n = sd_idx.loc[(2008, t2), 'SeedNum']
        except KeyError:
            continue

        row = {f'diff_{col}': f1[col] - f2[col] for col in STAT_COLS[:-1]}
        row['diff_SeedNum'] = s1n - s2n

        prob = rf.predict_proba(pd.DataFrame([row])[DIFF_COLS])[0, 1]
        sub_rows.append({'ID': f'2008_{t1}_{t2}', 'Pred': round(prob, 6)})

submission = pd.DataFrame(sub_rows)
submission.to_csv('/Users/shaurya/Downloads/submission_2008.csv', index=False)
print(f"\n✅ Submission saved → Downloads/submission_2008.csv  ({len(submission)} matchups)")
print(submission.head(10).to_string(index=False))


Team features shape: (8346, 16)
Skipped 0 games (missing team/seed data)

Total matchup rows    : 2898
Seasons in matchup df : [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Label distribution    :
Label
1    1449
0    1449
Name: count, dtype: int64

Train rows : 640  (seasons [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007)])
Test rows  : 128   (season 2008)

Model trained ✅

── 2008 Tournament Evaluation ──
Accuracy  : 0.7734
Log Loss  : 0.4966  (Kaggle metric — baseline random = 0.693)

── Top 10 Features ──
           Feature  Importance
      diff_SeedNum    0.290727
 diff_AvgPointDiff    0.154443
      diff_WinRate    0.082500
     diff_AvgFGpct   

In [1]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║  Random Forest  │  Predict 2023 NCAA Tournament                     ║
║                                                                      ║
║  Identical data pipeline to XGBoost 2023 model — only the model    ║
║  is swapped to scikit-learn RandomForestClassifier for a direct     ║
║  apples-to-apples comparison.                                        ║
║                                                                      ║
║  TRAINING DATA (labels = game outcomes):                             ║
║    ① Regular season games  2013–2023  → each game = one training row ║
║       features = game-level box-score differentials                  ║
║    ② Tournament games      2013–2022  → each game = one training row ║
║       features = that season's reg-season aggregated stats + seeds   ║
║                                                                      ║
║  TEST DATA (never touched during training):                          ║
║    2023 Tournament matchups only                                     ║
║    Features: 2023 regular-season aggregated stats + 2023 seeds       ║
║                                                                      ║
║  NO ELO — box-score + seed features only                             ║
║                                                                      ║
║  NaN HANDLING                                                        ║
║    Seeds / WinRate / GamesDiff are NaN for regular-season rows.     ║
║    scikit-learn RandomForest does NOT natively handle NaN —          ║
║    these columns are median-imputed separately for:                  ║
║      • regular-season rows  (no seed / winrate context)              ║
║      • tournament rows      (seeds and winrate are known)            ║
║    Imputation uses only training-set statistics (no test leakage).  ║
║                                                                      ║
║  LEAKAGE CONTROLS                                                    ║
║    • Reg-season rows use ONLY that game's own box score.            ║
║    • Tournament feature rows use ONLY same-season reg-season stats  ║
║      (computed before the tournament starts each year).             ║
║    • 2023 tournament outcomes are NEVER used in training.           ║
║    • 2023 regular-season data is used ONLY to build test features.  ║
║    • Imputation medians computed on training data only.             ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, roc_auc_score
import sklearn

print(f"scikit-learn version: {sklearn.__version__}")

# ─────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')
m_seeds                   = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

print("Regular Season shape :", m_regular_season_detailed.shape)
print("Seeds shape          :", m_seeds.shape)
print("Tourney Results shape:", m_tourney_detailed_result.shape)

# ─────────────────────────────────────────────────────────────────
# 2. PARSE SEEDS  (e.g. "W01" → 1, "Z16a" → 16)
# ─────────────────────────────────────────────────────────────────
def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

# ─────────────────────────────────────────────────────────────────
# 3. SEASON-AGGREGATED TEAM STATS
#    Built per season from regular-season games only.
#    Used as features for tournament matchup rows.
# ─────────────────────────────────────────────────────────────────
def compute_team_stats(reg_df, seasons):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()

    win = rows.rename(columns={
        'WTeamID':'TeamID', 'LTeamID':'OppID',
        'WScore':'Pts',     'LScore':'OppPts',
        'WFGM':'FGM','WFGA':'FGA','WFGM3':'FGM3','WFGA3':'FGA3',
        'WFTM':'FTM','WFTA':'FTA','WOR':'OR','WDR':'DR',
        'WAst':'Ast','WTO':'TO','WStl':'Stl','WBlk':'Blk','WPF':'PF',
    }).copy()
    win['Win'] = 1

    lose = rows.rename(columns={
        'LTeamID':'TeamID', 'WTeamID':'OppID',
        'LScore':'Pts',     'WScore':'OppPts',
        'LFGM':'FGM','LFGA':'FGA','LFGM3':'FGM3','LFGA3':'FGA3',
        'LFTM':'FTM','LFTA':'FTA','LOR':'OR','LDR':'DR',
        'LAst':'Ast','LTO':'TO','LStl':'Stl','LBlk':'Blk','LPF':'PF',
    }).copy()
    lose['Win'] = 0

    keep = ['Season','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3',
            'FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF','Win']
    all_g = pd.concat([win[keep], lose[keep]], ignore_index=True)

    agg = all_g.groupby(['Season','TeamID']).agg(
        Games      =('Win','count'),
        WinRate    =('Win','mean'),
        AvgPts     =('Pts','mean'),
        AvgOppPts  =('OppPts','mean'),
        AvgFGM     =('FGM','mean'),
        AvgFGA     =('FGA','mean'),
        AvgFGM3    =('FGM3','mean'),
        AvgFGA3    =('FGA3','mean'),
        AvgFTM     =('FTM','mean'),
        AvgFTA     =('FTA','mean'),
        AvgOR      =('OR','mean'),
        AvgDR      =('DR','mean'),
        AvgAst     =('Ast','mean'),
        AvgTO      =('TO','mean'),
        AvgStl     =('Stl','mean'),
        AvgBlk     =('Blk','mean'),
        AvgPF      =('PF','mean'),
    ).reset_index()

    agg['FGPct']  = agg['AvgFGM']  / agg['AvgFGA'].replace(0, np.nan)
    agg['FG3Pct'] = agg['AvgFGM3'] / agg['AvgFGA3'].replace(0, np.nan)
    agg['FTPct']  = agg['AvgFTM']  / agg['AvgFTA'].replace(0, np.nan)
    agg['Margin'] = agg['AvgPts']  - agg['AvgOppPts']
    return agg.set_index(['Season','TeamID'])


# ─────────────────────────────────────────────────────────────────
# 4A. REGULAR-SEASON TRAINING ROWS
#     One row per game. Features = that game's raw box-score diffs.
#     Seeds / WinRate / GamesDiff are NaN for these rows.
# ─────────────────────────────────────────────────────────────────
def build_reg_season_train_rows(reg_df, seasons):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()
    records = []

    def pct(m, a):
        return m / a if a and a > 0 else np.nan

    for _, g in rows.iterrows():
        w_id, l_id = int(g['WTeamID']), int(g['LTeamID'])

        if w_id < l_id:
            t_a, t_b, label  = w_id, l_id, 1
            pts_a, pts_b     = g['WScore'], g['LScore']
            fgm_a, fga_a     = g['WFGM'],  g['WFGA']
            fgm3_a, fga3_a   = g['WFGM3'], g['WFGA3']
            ftm_a, fta_a     = g['WFTM'],  g['WFTA']
            or_a, dr_a       = g['WOR'],   g['WDR']
            ast_a, to_a      = g['WAst'],  g['WTO']
            stl_a, blk_a, pf_a = g['WStl'], g['WBlk'], g['WPF']
            fgm_b, fga_b     = g['LFGM'],  g['LFGA']
            fgm3_b, fga3_b   = g['LFGM3'], g['LFGA3']
            ftm_b, fta_b     = g['LFTM'],  g['LFTA']
            or_b, dr_b       = g['LOR'],   g['LDR']
            ast_b, to_b      = g['LAst'],  g['LTO']
            stl_b, blk_b, pf_b = g['LStl'], g['LBlk'], g['LPF']
        else:
            t_a, t_b, label  = l_id, w_id, 0
            pts_a, pts_b     = g['LScore'], g['WScore']
            fgm_a, fga_a     = g['LFGM'],  g['LFGA']
            fgm3_a, fga3_a   = g['LFGM3'], g['LFGA3']
            ftm_a, fta_a     = g['LFTM'],  g['LFTA']
            or_a, dr_a       = g['LOR'],   g['LDR']
            ast_a, to_a      = g['LAst'],  g['LTO']
            stl_a, blk_a, pf_a = g['LStl'], g['LBlk'], g['LPF']
            fgm_b, fga_b     = g['WFGM'],  g['WFGA']
            fgm3_b, fga3_b   = g['WFGM3'], g['WFGA3']
            ftm_b, fta_b     = g['WFTM'],  g['WFTA']
            or_b, dr_b       = g['WOR'],   g['WDR']
            ast_b, to_b      = g['WAst'],  g['WTO']
            stl_b, blk_b, pf_b = g['WStl'], g['WBlk'], g['WPF']

        fg_a  = pct(fgm_a, fga_a);   fg_b  = pct(fgm_b, fga_b)
        fg3_a = pct(fgm3_a, fga3_a); fg3_b = pct(fgm3_b, fga3_b)
        ft_a  = pct(ftm_a, fta_a);   ft_b  = pct(ftm_b, fta_b)
        margin_a = pts_a - pts_b

        records.append({
            'Season': int(g['Season']), 'TeamA': t_a, 'TeamB': t_b,
            'Label': label, 'DataSource': 'RegularSeason',

            # NaN for reg-season — will be imputed from training medians
            'SeedA': np.nan, 'SeedB': np.nan, 'SeedDiff': np.nan,

            # Difference features (A − B)
            'WinRateDiff': np.nan,
            'PtsDiff':     pts_a - pts_b,
            'OppPtsDiff':  pts_b - pts_a,
            'MarginDiff':  margin_a,
            'FGPctDiff':   (fg_a  or 0) - (fg_b  or 0),
            'FG3PctDiff':  (fg3_a or 0) - (fg3_b or 0),
            'FTPctDiff':   (ft_a  or 0) - (ft_b  or 0),
            'ORDiff':      or_a  - or_b,
            'DRDiff':      dr_a  - dr_b,
            'AstDiff':     ast_a - ast_b,
            'TODiff':      to_a  - to_b,
            'StlDiff':     stl_a - stl_b,
            'BlkDiff':     blk_a - blk_b,
            'PFDiff':      pf_a  - pf_b,
            'GamesDiff':   np.nan,

            # Raw Team A
            'A_WinRate': np.nan,   'A_AvgPts': pts_a,   'A_Margin': margin_a,
            'A_FGPct':   fg_a,     'A_FG3Pct': fg3_a,   'A_FTPct':  ft_a,
            'A_AvgOR':   or_a,     'A_AvgDR':  dr_a,
            'A_AvgAst':  ast_a,    'A_AvgTO':  to_a,
            'A_AvgStl':  stl_a,    'A_AvgBlk': blk_a,

            # Raw Team B
            'B_WinRate': np.nan,   'B_AvgPts': pts_b,   'B_Margin': -margin_a,
            'B_FGPct':   fg_b,     'B_FG3Pct': fg3_b,   'B_FTPct':  ft_b,
            'B_AvgOR':   or_b,     'B_AvgDR':  dr_b,
            'B_AvgAst':  ast_b,    'B_AvgTO':  to_b,
            'B_AvgStl':  stl_b,    'B_AvgBlk': blk_b,
        })

    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────
# 4B. TOURNAMENT MATCHUP ROWS
#     Features = season-agg reg-season stats + seeds.
#     Used for training (2013–2022) and test (2023).
# ─────────────────────────────────────────────────────────────────
def build_tourney_matchup_rows(tourney_df, stats_df, seeds_df, seasons):
    records = []
    for _, row in tourney_df[tourney_df['Season'].isin(seasons)].iterrows():
        season = row['Season']
        w_id, l_id = row['WTeamID'], row['LTeamID']

        if w_id < l_id:
            t_a, t_b, label = w_id, l_id, 1
        else:
            t_a, t_b, label = l_id, w_id, 0

        if (season, t_a) not in stats_df.index or (season, t_b) not in stats_df.index:
            continue

        sa = stats_df.loc[(season, t_a)]
        sb = stats_df.loc[(season, t_b)]

        s_a = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_a)]
        s_b = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_b)]
        seed_a = s_a['SeedNum'].values[0] if len(s_a) else np.nan
        seed_b = s_b['SeedNum'].values[0] if len(s_b) else np.nan

        records.append({
            'Season': season, 'TeamA': t_a, 'TeamB': t_b,
            'Label': label, 'DataSource': 'Tournament',

            'SeedA': seed_a, 'SeedB': seed_b, 'SeedDiff': seed_a - seed_b,

            'WinRateDiff':  sa['WinRate']  - sb['WinRate'],
            'PtsDiff':      sa['AvgPts']   - sb['AvgPts'],
            'OppPtsDiff':   sa['AvgOppPts']- sb['AvgOppPts'],
            'MarginDiff':   sa['Margin']   - sb['Margin'],
            'FGPctDiff':    sa['FGPct']    - sb['FGPct'],
            'FG3PctDiff':   sa['FG3Pct']   - sb['FG3Pct'],
            'FTPctDiff':    sa['FTPct']    - sb['FTPct'],
            'ORDiff':       sa['AvgOR']    - sb['AvgOR'],
            'DRDiff':       sa['AvgDR']    - sb['AvgDR'],
            'AstDiff':      sa['AvgAst']   - sb['AvgAst'],
            'TODiff':       sa['AvgTO']    - sb['AvgTO'],
            'StlDiff':      sa['AvgStl']   - sb['AvgStl'],
            'BlkDiff':      sa['AvgBlk']   - sb['AvgBlk'],
            'PFDiff':       sa['AvgPF']    - sb['AvgPF'],
            'GamesDiff':    sa['Games']    - sb['Games'],

            'A_WinRate': sa['WinRate'], 'A_AvgPts': sa['AvgPts'],
            'A_Margin':  sa['Margin'],  'A_FGPct':  sa['FGPct'],
            'A_FG3Pct':  sa['FG3Pct'], 'A_FTPct':  sa['FTPct'],
            'A_AvgOR':   sa['AvgOR'],   'A_AvgDR':  sa['AvgDR'],
            'A_AvgAst':  sa['AvgAst'], 'A_AvgTO':   sa['AvgTO'],
            'A_AvgStl':  sa['AvgStl'], 'A_AvgBlk':  sa['AvgBlk'],

            'B_WinRate': sb['WinRate'], 'B_AvgPts': sb['AvgPts'],
            'B_Margin':  sb['Margin'],  'B_FGPct':  sb['FGPct'],
            'B_FG3Pct':  sb['FG3Pct'], 'B_FTPct':  sb['FTPct'],
            'B_AvgOR':   sb['AvgOR'],   'B_AvgDR':  sb['AvgDR'],
            'B_AvgAst':  sb['AvgAst'], 'B_AvgTO':   sb['AvgTO'],
            'B_AvgStl':  sb['AvgStl'], 'B_AvgBlk':  sb['AvgBlk'],
        })
    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────
# 5. ASSEMBLE TRAIN / TEST
# ─────────────────────────────────────────────────────────────────
REG_TRAIN_SEASONS   = list(range(2013, 2024))   # 2013–2023 regular seasons
TOURN_TRAIN_SEASONS = list(range(2013, 2023))   # 2013–2022 tournaments
TEST_SEASONS        = [2023]

stats = compute_team_stats(m_regular_season_detailed, list(range(2013, 2024)))

reg_train_df   = build_reg_season_train_rows(m_regular_season_detailed, REG_TRAIN_SEASONS)
tourn_train_df = build_tourney_matchup_rows(m_tourney_detailed_result, stats, m_seeds, TOURN_TRAIN_SEASONS)
train_df       = pd.concat([reg_train_df, tourn_train_df], ignore_index=True)
test_df        = build_tourney_matchup_rows(m_tourney_detailed_result, stats, m_seeds, TEST_SEASONS)

print(f"\n{'─'*55}")
print(f"  Training rows")
print(f"{'─'*55}")
print(f"  Regular season games  2013–2023 : {len(reg_train_df):>7,}")
print(f"  Tournament games      2013–2022 : {len(tourn_train_df):>7,}")
print(f"  TOTAL                           : {len(train_df):>7,}")
print(f"\n  Test rows (2023 tournament)     : {len(test_df):>7,}")
print(f"{'─'*55}")

# ─────────────────────────────────────────────────────────────────
# 6. FEATURE COLUMNS  (same 42 as XGBoost 2023 model)
# ─────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    'SeedA', 'SeedB', 'SeedDiff',
    'WinRateDiff', 'PtsDiff', 'OppPtsDiff', 'MarginDiff',
    'FGPctDiff', 'FG3PctDiff', 'FTPctDiff',
    'ORDiff', 'DRDiff', 'AstDiff', 'TODiff', 'StlDiff', 'BlkDiff', 'PFDiff',
    'GamesDiff',
    'A_WinRate', 'A_AvgPts', 'A_Margin', 'A_FGPct', 'A_FG3Pct', 'A_FTPct',
    'A_AvgOR', 'A_AvgDR', 'A_AvgAst', 'A_AvgTO', 'A_AvgStl', 'A_AvgBlk',
    'B_WinRate', 'B_AvgPts', 'B_Margin', 'B_FGPct', 'B_FG3Pct', 'B_FTPct',
    'B_AvgOR', 'B_AvgDR', 'B_AvgAst', 'B_AvgTO', 'B_AvgStl', 'B_AvgBlk',
]

X_train_raw = train_df[FEATURE_COLS]
y_train     = train_df['Label']
X_test_raw  = test_df[FEATURE_COLS]
y_test      = test_df['Label']

print(f"\n  Features : {len(FEATURE_COLS)}")
print(f"  NaN cols (Seed*, WinRate*, GamesDiff) → median-imputed from training data")

# ─────────────────────────────────────────────────────────────────
# 7. NaN IMPUTATION
#    scikit-learn RF requires no NaN values.
#    Imputer fitted on training data ONLY — no test leakage.
#    Strategy: median (robust to outliers, consistent with tree logic).
# ─────────────────────────────────────────────────────────────────
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train_raw)   # fit + transform on train
X_test  = imputer.transform(X_test_raw)        # transform only on test

# Report imputed medians for the NaN-heavy columns
nan_cols = ['SeedA','SeedB','SeedDiff','WinRateDiff','A_WinRate','B_WinRate','GamesDiff']
nan_idxs = [FEATURE_COLS.index(c) for c in nan_cols]
print("\n  Imputed medians (training set):")
for col, idx in zip(nan_cols, nan_idxs):
    print(f"    {col:<16s} → {imputer.statistics_[idx]:.4f}")

# ─────────────────────────────────────────────────────────────────
# 8. TRAIN RANDOM FOREST
#    Hyperparameters tuned to balance bias/variance on this dataset:
#    - n_estimators=500: enough trees for stable probability estimates
#    - max_depth=12: deep enough to capture interactions, not so deep
#      it memorises regular-season noise
#    - min_samples_leaf=20: matches XGBoost's min_child_weight=20
#      to prevent overfitting on small tournament-only subsets
#    - max_features='sqrt': standard RF feature sampling per split
#    - class_weight='balanced': handles slight label imbalance from
#      mirroring (each game appears once, winner is always TeamA=1
#      in ~50% of rows by construction — so balanced is conservative)
#    - n_jobs=-1: parallelise across all CPU cores
# ─────────────────────────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators   = 500,
    max_depth      = 12,
    min_samples_leaf= 20,
    max_features   = 'sqrt',
    class_weight   = 'balanced',
    random_state   = 42,
    n_jobs         = -1,
    oob_score      = True,   # out-of-bag estimate (free internal validation)
)

print("\nTraining Random Forest (n_estimators=500)...")
model.fit(X_train, y_train)
print(f"  OOB Score (accuracy) : {model.oob_score_:.4f}")

# ─────────────────────────────────────────────────────────────────
# 9. EVALUATE ON 2023 TOURNAMENT
# ─────────────────────────────────────────────────────────────────
y_pred_prob = model.predict_proba(X_test)[:, 1]
y_pred_bin  = (y_pred_prob >= 0.5).astype(int)

ll  = log_loss(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
acc = (y_pred_bin == y_test.values).mean()

print(f"\n{'='*55}")
print(f"  Random Forest — 2023 Tournament Evaluation")
print(f"  Train: reg season 2013-2023 + tournaments 2013-2022")
print(f"  Test : 2023 tournament only")
print(f"{'='*55}")
print(f"  Log Loss : {ll:.4f}   (random baseline = 0.6931)")
print(f"  ROC AUC  : {auc:.4f}   (random baseline = 0.5000)")
print(f"  Accuracy : {acc:.4f}   ({int(acc*len(y_test))}/{len(y_test)} correct)")
print(f"  OOB Score: {model.oob_score_:.4f}   (internal train-set estimate)")

# ─────────────────────────────────────────────────────────────────
# 10. FEATURE IMPORTANCE  (mean decrease in impurity — Gini)
#     Standard RF importance: different from XGBoost gain but
#     both measure how much each feature reduces node impurity.
# ─────────────────────────────────────────────────────────────────
importance = pd.DataFrame({
    'Feature':    FEATURE_COLS,
    'Importance': model.feature_importances_,
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nTop 15 Features by Mean Decrease in Impurity (Gini):")
print(importance.head(15).to_string(index=False))

# ─────────────────────────────────────────────────────────────────
# 11. OUTPUT
# ─────────────────────────────────────────────────────────────────
results_df = test_df[['Season','TeamA','TeamB','Label'] + FEATURE_COLS].copy()
results_df['PredProb_TeamA_Wins'] = y_pred_prob
results_df['PredWinner']   = results_df.apply(
    lambda r: r['TeamA'] if r['PredProb_TeamA_Wins'] >= 0.5 else r['TeamB'], axis=1)
results_df['ActualWinner'] = results_df.apply(
    lambda r: r['TeamA'] if r['Label'] == 1 else r['TeamB'], axis=1)
results_df['Correct']      = (results_df['PredWinner'] == results_df['ActualWinner']).astype(int)
results_df['FeaturesUsed'] = ' | '.join(FEATURE_COLS)

results_df.to_csv('/Users/shaurya/Downloads/rf_2023_predictions.csv', index=False)
importance.to_csv('/Users/shaurya/Downloads/rf_2023_feature_importance.csv', index=False)

print(f"\n✅  Predictions saved  → /Users/shaurya/Downloads/rf_2023_predictions.csv")
print(f"✅  Importance saved   → /Users/shaurya/Downloads/rf_2023_feature_importance.csv")
print(f"\nTotal features : {len(FEATURE_COLS)}")
print("Feature list   :", FEATURE_COLS)

scikit-learn version: 1.8.0
Regular Season shape : (122775, 34)
Seeds shape          : (2626, 3)
Tourney Results shape: (1449, 34)

───────────────────────────────────────────────────────
  Training rows
───────────────────────────────────────────────────────
  Regular season games  2013–2023 :  57,798
  Tournament games      2013–2022 :     602
  TOTAL                           :  58,400

  Test rows (2023 tournament)     :      67
───────────────────────────────────────────────────────

  Features : 42
  NaN cols (Seed*, WinRate*, GamesDiff) → median-imputed from training data

  Imputed medians (training set):
    SeedA            → 6.5000
    SeedB            → 7.0000
    SeedDiff         → 0.0000
    WinRateDiff      → 0.0110
    A_WinRate        → 0.7500
    B_WinRate        → 0.7419
    GamesDiff        → 0.0000

Training Random Forest (n_estimators=500)...
  OOB Score (accuracy) : 0.9966

  Random Forest — 2023 Tournament Evaluation
  Train: reg season 2013-2023 + tournaments 2